# Selection Overfit Audit v1

Notebook-owned model-selection audit for the active objective: reduce overfit, keep excess return above +30pp, and move MaxDD toward the hard 15% target before opening another training branch.

Governance:
- Baseline universe: current repo model summaries plus strict daily artifacts under `runtime/model/card`, `runtime/execution/card`, and `runtime/production/card`
- Direct comparison baselines: `v19reg_v22_budget_usage_bull_guard_v1`, `recovery_reentry_regime_drawdown_guard_v1`, and `risk_budget_usage_v1_actor_parity`
- Closest historical success: walk-forward-aware runtime selection (`v22_post_release_cap_runtime_auto_select_v1`) and the v19_reg/v22 budget-usage composer
- Closest historical failure: continued v22 training and direct SOXX feature injection, both of which improved isolated metrics while worsening OOS risk or walk-forward behavior
- Why this is not a duplicate: this is not a new model, reward, factor, or threshold. It audits the existing search surface and tests a conservative risk-budget frontier before retraining.
- Role: `diagnose`
- Changed layer: `selection`
- Target: identify whether any existing artifact or conservative risk-budget rescaling reaches `relative_return >= +30pp` and `MaxDD <= 15%`; if none, identify the closest validated frontier and next non-duplicate mechanism
- Notebook rule: this `.ipynb` is the primary single-run conclusion artifact

External basis:
- Probability of Backtest Overfitting / Deflated Sharpe Ratio: https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf
- Backtest overfitting in strategy selection: https://academic.oup.com/jrssig/article/18/6/22/7038278
- GT-Score robust objective for reducing overfitting: https://arxiv.org/abs/2602.00080
- Qlib research-to-execution framework: https://github.com/microsoft/qlib
- FinRL risk-aware portfolio management framework: https://github.com/AI4Finance-Foundation/FinRL


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = 'src:.'

CARD = 'selection_overfit_audit_v1'
CARD_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / CARD
CARD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
STRICT_START = pd.Timestamp('2020-01-02')
STRICT_END = pd.Timestamp('2026-05-21')

pd.set_option('display.max_columns', 80)
print('card_dir', CARD_DIR.relative_to(ROOT))


card_dir runtime/model_selection/card/selection_overfit_audit_v1


In [2]:
def _safe_float(value, default=np.nan) -> float:
    try:
        if value is None:
            return default
        return float(value)
    except Exception:
        return default


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def selection_history_stats(history: list[dict]) -> dict:
    if not history:
        return {
            'history_len': 0,
            'best_iter': np.nan,
            'last_iter': np.nan,
            'best_history_score': np.nan,
            'last_history_score': np.nan,
            'peak_not_last': False,
            'peak_to_last_score_drop': np.nan,
        }
    scores = np.array([_safe_float(row.get('score')) for row in history], dtype=float)
    iters = [row.get('iter') for row in history]
    best_idx = int(np.nanargmax(scores))
    last_idx = len(history) - 1
    return {
        'history_len': int(len(history)),
        'best_iter': iters[best_idx],
        'last_iter': iters[last_idx],
        'best_history_score': float(scores[best_idx]),
        'last_history_score': float(scores[last_idx]),
        'peak_not_last': bool(best_idx != last_idx),
        'peak_to_last_score_drop': float(scores[best_idx] - scores[last_idx]),
    }


def metric_get(metrics: dict | None, name: str) -> float:
    return _safe_float((metrics or {}).get(name))


def extract_summary_row(path: Path) -> dict:
    payload = load_json(path)
    profile = payload.get('profile') or path.parent.name
    valid = payload.get('valid') or {}
    proxy = payload.get('test_proxy') or {}
    slices = payload.get('slices') or {}
    test_2020 = slices.get('test_2020') or {}
    test_2022 = slices.get('test_2022') or {}
    test_2023_2024 = slices.get('test_2023_2024') or {}
    promotion = payload.get('promotion') or {}
    wf = promotion.get('walk_forward') or {}
    fixed = promotion.get('fixed_split') or {}
    stats = selection_history_stats(payload.get('selection_history') or [])
    valid_sharpe = metric_get(valid, 'sharpe')
    proxy_sharpe = metric_get(proxy, 'sharpe')
    row = {
        'profile': profile,
        'summary_path': str(path.relative_to(ROOT)),
        'best_score': _safe_float(payload.get('best_score')),
        'valid_return': metric_get(valid, 'return'),
        'valid_sharpe': valid_sharpe,
        'valid_max_dd': metric_get(valid, 'max_dd'),
        'valid_avg_pos': metric_get(valid, 'avg_pos'),
        'proxy_return': metric_get(proxy, 'return'),
        'proxy_relative_return': metric_get(proxy, 'relative_return'),
        'proxy_sharpe': proxy_sharpe,
        'proxy_max_dd': metric_get(proxy, 'max_dd'),
        'proxy_avg_pos': metric_get(proxy, 'avg_pos'),
        'test_2020_return': metric_get(test_2020, 'return'),
        'test_2020_max_dd': metric_get(test_2020, 'max_dd'),
        'test_2022_return': metric_get(test_2022, 'return'),
        'test_2022_relative_return': metric_get(test_2022, 'relative_return'),
        'test_2022_max_dd': metric_get(test_2022, 'max_dd'),
        'test_2022_avg_pos': metric_get(test_2022, 'avg_pos'),
        'test_2023_2024_return': metric_get(test_2023_2024, 'return'),
        'test_2023_2024_relative_return': metric_get(test_2023_2024, 'relative_return'),
        'test_2023_2024_max_dd': metric_get(test_2023_2024, 'max_dd'),
        'promotion_eligible': bool(promotion.get('eligible', False)),
        'fixed_split_eligible': bool(fixed.get('eligible', False)),
        'walk_forward_eligible': bool(wf.get('eligible', False)),
        'walk_forward_status': wf.get('status', 'present' if wf else 'missing'),
        'valid_proxy_sharpe_gap': valid_sharpe - proxy_sharpe,
    }
    row.update(stats)
    return row

summary_paths = sorted((ROOT / 'runtime' / 'model' / 'card').glob('**/*summary.json'))
summary_rows = []
for path in summary_paths:
    try:
        summary_rows.append(extract_summary_row(path))
    except Exception as exc:
        summary_rows.append({'profile': path.parent.name, 'summary_path': str(path.relative_to(ROOT)), 'error': str(exc)})
summary_table = pd.DataFrame(summary_rows)
summary_table['proxy_target_pass'] = (summary_table['proxy_relative_return'] >= TARGET_RELATIVE_RETURN) & (summary_table['proxy_max_dd'] <= TARGET_MAX_DD)
summary_table['fixed_2022_risk_fail'] = (summary_table['test_2022_max_dd'] > 35.0) | (summary_table['test_2022_avg_pos'] > 0.35)
summary_table['selection_overfit_flag'] = summary_table['peak_not_last'].fillna(False) & (summary_table['peak_to_last_score_drop'].fillna(0.0) >= 5.0)
summary_table['valid_proxy_gap_flag'] = summary_table['valid_proxy_sharpe_gap'].fillna(0.0) >= 0.50
summary_table['selection_risk_score'] = (
    summary_table['selection_overfit_flag'].astype(float) * 2.0
    + summary_table['valid_proxy_gap_flag'].astype(float) * 1.5
    + summary_table['fixed_2022_risk_fail'].astype(float) * 2.0
    + (~summary_table['walk_forward_eligible']).astype(float) * 1.0
)
summary_table_path = CARD_DIR / 'summary_selection_overfit_table.csv'
summary_table.sort_values(['selection_risk_score', 'best_score'], ascending=[False, False]).to_csv(summary_table_path, index=False)
print('summary rows', len(summary_table), 'proxy_target_pass', int(summary_table['proxy_target_pass'].sum()), 'selection_overfit_flags', int(summary_table['selection_overfit_flag'].sum()))
summary_table.sort_values(['selection_risk_score', 'best_score'], ascending=[False, False]).head(12)[[
    'profile', 'best_score', 'best_iter', 'last_iter', 'peak_to_last_score_drop', 'proxy_relative_return',
    'proxy_max_dd', 'test_2022_max_dd', 'test_2022_avg_pos', 'walk_forward_eligible', 'selection_risk_score',
]]


summary rows 94 proxy_target_pass 0 selection_overfit_flags 55


,profile,best_score,best_iter,last_iter,peak_to_last_score_drop,proxy_relative_return,proxy_max_dd,test_2022_max_dd,test_2022_avg_pos,walk_forward_eligible,selection_risk_score
10,balanced_floor40_sym_expectancy,112.832268,0,40,33.078441,NaN,65.047914,49.600911,0.437207,False,6.5
11,balanced_floor40_sym_expectancy,112.832268,0,80,41.626778,NaN,65.047914,49.600911,0.437207,False,6.5
12,balanced_floor40_sym,99.326365,20,80,9.091648,NaN,74.388199,60.555678,0.537771,False,6.5
69,balanced_rule_residual_v19_small,96.421545,0,80,29.846925,NaN,38.253942,38.253942,0.307105,False,6.5
4,balanced_floor40,94.262048,20,80,20.036167,NaN,65.680650,51.358924,0.477719,False,6.5
78,balanced_rule_residual_v22_release_v1,90.915946,0,40,8.544050,-666.583561,60.350126,38.845207,0.375263,False,6.5
67,balanced_rule_residual_v19_small,90.059305,0,80,23.102213,NaN,55.072229,38.253942,0.307105,False,6.5
63,balanced_rule_residual_v19_reg_v22_retest_v1,89.478679,0,80,18.645857,-744.760860,59.637288,58.299843,0.380930,False,6.5
79,balanced_rule_residual_v22_soxx,89.478679,0,80,13.076419,-565.449207,61.093959,44.153001,0.380930,False,6.5
77,balanced_rule_residual_v22_continued_retention_v1,86.241715,0,20,16.984689,-1084.294474,59.130395,55.255626,0.321447,False,6.5


In [3]:
def metrics_from_daily(path: Path) -> dict | None:
    try:
        frame = pd.read_csv(path)
    except Exception:
        return None
    required = {'datetime', 'position', 'price'}
    if not required.issubset(frame.columns):
        return None
    frame['datetime'] = pd.to_datetime(frame['datetime'], errors='coerce')
    frame = frame.dropna(subset=['datetime', 'position', 'price']).sort_values('datetime')
    frame = frame[(frame['datetime'] >= STRICT_START) & (frame['datetime'] <= STRICT_END)]
    if len(frame) < 200:
        return None
    positions = frame['position'].astype(float).to_numpy()
    prices = frame['price'].astype(float).to_numpy()
    returns = np.diff(prices) / prices[:-1]
    if len(returns) < 2:
        return None
    strategy_returns = positions[:-1] * returns
    nav = np.cumprod(1.0 + strategy_returns)
    buyhold_nav = np.cumprod(1.0 + returns)
    drawdowns = nav / np.maximum.accumulate(nav) - 1.0
    total_return = float(nav[-1] - 1.0) * 100.0
    buyhold_return = float(buyhold_nav[-1] - 1.0) * 100.0
    sharpe = float(strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)) if strategy_returns.std() > 0 else 0.0
    skew = float(pd.Series(strategy_returns).skew())
    kurt = float(pd.Series(strategy_returns).kurt() + 3.0)
    daily_sr = sharpe / np.sqrt(252)
    denom = max(1e-12, 1.0 - skew * daily_sr + ((kurt - 1.0) / 4.0) * daily_sr * daily_sr)
    psr_z = daily_sr * math.sqrt(max(len(strategy_returns) - 1, 1)) / math.sqrt(denom)
    psr_positive = 0.5 * (1.0 + math.erf(psr_z / math.sqrt(2.0)))
    trough_idx = int(np.argmin(drawdowns))
    peak_idx = int(np.argmax(nav[: trough_idx + 1]))
    return {
        'card': path.parent.name,
        'path': str(path.relative_to(ROOT)),
        'rows': int(len(frame)),
        'start': frame['datetime'].iloc[0].date().isoformat(),
        'end': frame['datetime'].iloc[-1].date().isoformat(),
        'return': total_return,
        'buyhold_return': buyhold_return,
        'relative_return': total_return - buyhold_return,
        'sharpe': sharpe,
        'max_dd': abs(float(drawdowns.min()) * 100.0),
        'avg_pos': float(positions.mean()),
        'trades': int(np.sum(np.abs(np.diff(positions)) > 0.001)),
        'psr_positive': psr_positive,
        'skew': skew,
        'kurtosis': kurt,
        'maxdd_peak': frame['datetime'].iloc[peak_idx].date().isoformat(),
        'maxdd_trough': frame['datetime'].iloc[trough_idx + 1 if trough_idx + 1 < len(frame) else trough_idx].date().isoformat(),
    }


def candidate_name_from_path(path: Path) -> str:
    parts = path.relative_to(ROOT).parts
    if 'card' in parts:
        idx = parts.index('card')
        if idx + 1 < len(parts):
            return parts[idx + 1]
    return path.parent.name

strict_rows = []
wf_rows = []
for path in sorted((ROOT / 'runtime').glob('**/*_soxl_daily.csv')):
    rel = path.relative_to(ROOT)
    path_text = str(rel)
    is_wf = 'walkforward' in path.parts or any(token in path.name for token in ['_wf_2020', '_wf_2022', '_wf_2023_2024'])
    metric = metrics_from_daily(path)
    if metric is None:
        continue
    metric['candidate'] = candidate_name_from_path(path)
    if is_wf:
        if 'wf_2020' in path_text:
            metric['wf'] = 'wf_2020'
        elif 'wf_2022' in path_text:
            metric['wf'] = 'wf_2022'
        elif 'wf_2023_2024' in path_text:
            metric['wf'] = 'wf_2023_2024'
        else:
            metric['wf'] = 'unknown'
        wf_rows.append(metric)
    else:
        if metric['rows'] >= 1000:
            strict_rows.append(metric)

strict_daily = pd.DataFrame(strict_rows)
strict_daily['target_pass'] = (strict_daily['relative_return'] >= TARGET_RELATIVE_RETURN) & (strict_daily['max_dd'] <= TARGET_MAX_DD)
strict_daily['target_gap'] = np.maximum(strict_daily['max_dd'] - TARGET_MAX_DD, 0.0) * 8.0 + np.maximum(TARGET_RELATIVE_RETURN - strict_daily['relative_return'], 0.0) * 2.0
strict_daily_path = CARD_DIR / 'strict_daily_target_screen.csv'
strict_daily.sort_values(['target_pass', 'target_gap', 'max_dd', 'relative_return'], ascending=[False, True, True, False]).to_csv(strict_daily_path, index=False)

wf_daily = pd.DataFrame(wf_rows)
wf_daily_path = CARD_DIR / 'walkforward_daily_screen.csv'
wf_daily.to_csv(wf_daily_path, index=False)
print('strict daily candidates', len(strict_daily), 'target_pass', int(strict_daily['target_pass'].sum()))
strict_daily.sort_values(['target_gap', 'max_dd', 'relative_return'], ascending=[True, True, False]).head(15)[[
    'candidate', 'card', 'return', 'relative_return', 'sharpe', 'max_dd', 'avg_pos', 'psr_positive', 'path'
]]


strict daily candidates 35 target_pass 1


,candidate,card,return,relative_return,sharpe,max_dd,avg_pos,psr_positive,path
11,risk_budget_residual_drawdown_guard_v1,risk_budget_residual_drawdown_guard_v1,176.604683,30.457960,0.971461,14.642461,0.160945,0.991927,runtime/execution/card/risk_budget_residual_dr...
7,risk_budget_crash_risk_factor_v1,risk_budget_crash_risk_factor_v1,190.745831,44.599109,1.002457,15.484927,0.163622,0.993554,runtime/execution/card/risk_budget_crash_risk_...
8,risk_budget_downside_vol_target_v1,risk_budget_downside_vol_target_v1,177.206488,31.059765,0.959340,16.277862,0.163318,0.991333,runtime/execution/card/risk_budget_downside_vo...
9,risk_budget_downside_vol_target_v1,risk_budget_downside_vol_target_v1,177.479796,31.333073,0.960142,16.277862,0.163345,0.991379,runtime/execution/card/risk_budget_downside_vo...
10,risk_budget_downside_vol_trendbreak_reentry_v1,risk_budget_downside_vol_trendbreak_reentry_v1,159.277092,13.130369,0.927840,15.406526,0.156982,0.989119,runtime/execution/card/risk_budget_downside_vo...
26,risk_budget_usage_v1,risk_budget_usage_v1,233.856985,87.710263,1.028952,21.375029,0.180393,0.995121,runtime/model/card/risk_budget_usage_v1/risk_b...
28,risk_budget_usage_v1,risk_budget_usage_v1,243.248590,97.101867,1.061732,23.408643,0.178202,0.996202,runtime/model/card/risk_budget_usage_v1/risk_b...
29,risk_budget_usage_v2,risk_budget_usage_v2,164.560804,18.414082,0.915031,20.929911,0.158132,0.989316,runtime/model/card/risk_budget_usage_v2/risk_b...
0,recovery_reentry_2021_trendbreak_cap_v1,recovery_reentry_2021_trendbreak_cap_v1,1023.710464,207.197964,1.080314,30.436657,0.408919,0.996759,runtime/execution/card/recovery_reentry_2021_t...
2,recovery_reentry_conditional_vol_target_v1,recovery_reentry_conditional_vol_target_v1,2988.234589,2171.722089,1.389946,30.923275,0.430109,0.999778,runtime/execution/card/recovery_reentry_condit...


In [4]:
def apply_scaled_floor(base_pos: np.ndarray, scale: float, floor: float) -> np.ndarray:
    return np.clip(base_pos * scale, floor, 1.0)


def apply_equity_drawdown_control(base_pos: np.ndarray, returns: np.ndarray, *, start: float, full: float, min_mult: float, recover: float, scale: float) -> np.ndarray:
    out = np.zeros_like(base_pos, dtype=float)
    nav = 1.0
    peak = 1.0
    for idx in range(len(base_pos)):
        dd = 1.0 - nav / peak
        if dd < recover:
            mult = 1.0
        elif dd <= start:
            mult = 1.0
        elif dd >= full:
            mult = min_mult
        else:
            span = max(full - start, 1e-9)
            mult = 1.0 - (1.0 - min_mult) * (dd - start) / span
        out[idx] = np.clip(base_pos[idx] * scale * mult, 0.0, 1.0)
        if idx < len(returns):
            nav *= 1.0 + out[idx] * returns[idx]
            peak = max(peak, nav)
    return out


def metric_from_position(frame: pd.DataFrame, positions: np.ndarray) -> dict:
    prices = frame['price'].astype(float).to_numpy()
    returns = np.diff(prices) / prices[:-1]
    strategy_returns = positions[:-1] * returns
    nav = np.cumprod(1.0 + strategy_returns)
    buyhold_nav = np.cumprod(1.0 + returns)
    dd = nav / np.maximum.accumulate(nav) - 1.0
    total_return = float(nav[-1] - 1.0) * 100.0
    buyhold_return = float(buyhold_nav[-1] - 1.0) * 100.0
    sharpe = float(strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)) if strategy_returns.std() > 0 else 0.0
    return {
        'return': total_return,
        'buyhold_return': buyhold_return,
        'relative_return': total_return - buyhold_return,
        'sharpe': sharpe,
        'max_dd': abs(float(dd.min()) * 100.0),
        'avg_pos': float(positions.mean()),
        'trades': int(np.sum(np.abs(np.diff(positions)) > 0.001)),
    }

risk_path = ROOT / 'runtime' / 'model' / 'card' / 'risk_budget_usage_v1' / 'risk_budget_usage_v1_actor_parity_soxl_daily.csv'
risk_frame = pd.read_csv(risk_path)
risk_frame['datetime'] = pd.to_datetime(risk_frame['datetime'])
risk_frame = risk_frame[(risk_frame['datetime'] >= STRICT_START) & (risk_frame['datetime'] <= STRICT_END)].dropna(subset=['position', 'price']).sort_values('datetime').reset_index(drop=True)
risk_pos = risk_frame['position'].astype(float).to_numpy()
risk_returns = np.diff(risk_frame['price'].astype(float).to_numpy()) / risk_frame['price'].astype(float).to_numpy()[:-1]
frontier_rows = []
for scale in np.linspace(0.1, 1.4, 27):
    for floor in [0.0, 0.02, 0.05, 0.08, 0.10, 0.12, 0.15]:
        pos = apply_scaled_floor(risk_pos, float(scale), float(floor))
        frontier_rows.append({'method': 'scaled_floor', 'scale': float(scale), 'floor': float(floor), **metric_from_position(risk_frame, pos)})
for start in [0.06, 0.08, 0.10, 0.12]:
    for full in [0.12, 0.15, 0.18, 0.20]:
        if full <= start:
            continue
        for min_mult in [0.0, 0.25, 0.50]:
            for recover in [0.02, 0.04, 0.06]:
                for scale in [0.8, 1.0, 1.2]:
                    pos = apply_equity_drawdown_control(risk_pos, risk_returns, start=start, full=full, min_mult=min_mult, recover=recover, scale=scale)
                    frontier_rows.append({
                        'method': 'equity_drawdown_control',
                        'scale': float(scale),
                        'floor': np.nan,
                        'start_drawdown': start,
                        'full_drawdown': full,
                        'min_mult': min_mult,
                        'recover': recover,
                        **metric_from_position(risk_frame, pos),
                    })
frontier = pd.DataFrame(frontier_rows)
frontier['target_pass'] = (frontier['relative_return'] >= TARGET_RELATIVE_RETURN) & (frontier['max_dd'] <= TARGET_MAX_DD)
frontier['target_gap'] = np.maximum(frontier['max_dd'] - TARGET_MAX_DD, 0.0) * 8.0 + np.maximum(TARGET_RELATIVE_RETURN - frontier['relative_return'], 0.0) * 2.0
frontier_path = CARD_DIR / 'risk_budget_conservative_frontier.csv'
frontier.sort_values(['target_pass', 'target_gap', 'max_dd', 'relative_return'], ascending=[False, True, True, False]).to_csv(frontier_path, index=False)
print('risk frontier rows', len(frontier), 'target_pass', int(frontier['target_pass'].sum()))
frontier.sort_values(['target_gap', 'max_dd', 'relative_return'], ascending=[True, True, False]).head(15)


risk frontier rows 594 target_pass 0


,method,scale,floor,return,buyhold_return,relative_return,sharpe,max_dd,avg_pos,trades,start_drawdown,full_drawdown,min_mult,recover,target_pass,target_gap
105,scaled_floor,0.85,0.00,183.515948,146.146723,37.369225,1.028952,18.381388,0.153334,1292,NaN,NaN,NaN,NaN,False,27.051102
106,scaled_floor,0.85,0.02,184.283184,146.146723,38.136461,1.031114,18.381388,0.154201,1292,NaN,NaN,NaN,NaN,False,27.051102
107,scaled_floor,0.85,0.05,185.357889,146.146723,39.211166,1.032098,18.468422,0.155936,1200,NaN,NaN,NaN,NaN,False,27.747375
100,scaled_floor,0.80,0.05,169.509463,146.146723,23.362740,1.030246,17.512973,0.147181,1133,NaN,NaN,NaN,NaN,False,33.378306
99,scaled_floor,0.80,0.02,168.843231,146.146723,22.696508,1.031226,17.366871,0.145181,1277,NaN,NaN,NaN,NaN,False,33.541955
397,equity_drawdown_control,1.00,NaN,178.410200,146.146723,32.263478,0.949810,19.205310,0.173770,1327,0.08,0.2,0.5,0.02,False,33.642478
400,equity_drawdown_control,1.00,NaN,178.410200,146.146723,32.263478,0.949810,19.205310,0.173770,1327,0.08,0.2,0.5,0.04,False,33.642478
403,equity_drawdown_control,1.00,NaN,178.410200,146.146723,32.263478,0.949810,19.205310,0.173770,1327,0.08,0.2,0.5,0.06,False,33.642478
108,scaled_floor,0.85,0.08,181.721121,146.146723,35.574398,1.004920,19.365443,0.162845,1031,NaN,NaN,NaN,NaN,False,34.923545
98,scaled_floor,0.80,0.00,168.117665,146.146723,21.970943,1.028952,17.366871,0.144314,1277,NaN,NaN,NaN,NaN,False,34.993086


In [5]:
strict_best = strict_daily.sort_values(['target_gap', 'max_dd', 'relative_return'], ascending=[True, True, False]).iloc[0]
strict_lowest_dd_rel_pass = strict_daily[strict_daily['relative_return'] >= TARGET_RELATIVE_RETURN].sort_values(['max_dd', 'relative_return'], ascending=[True, False]).iloc[0]
frontier_best = frontier.sort_values(['target_gap', 'max_dd', 'relative_return'], ascending=[True, True, False]).iloc[0]
frontier_rel_pass = frontier[frontier['relative_return'] >= TARGET_RELATIVE_RETURN].sort_values(['max_dd', 'relative_return'], ascending=[True, False]).iloc[0]
overfit_examples = summary_table[summary_table['selection_overfit_flag']].sort_values('peak_to_last_score_drop', ascending=False).head(8)
current_rows = summary_table[summary_table['profile'].isin([
    'balanced_rule_residual_v22',
    'balanced_rule_residual_v22_continued_retention_v1',
    'balanced_rule_residual_v19_reg',
    'balanced_rule_residual_v19_reg_light',
])].sort_values('profile')

summary_lines = []
for _, row in current_rows.iterrows():
    summary_lines.append(
        f"| `{row['profile']}` | {row['best_score']:.2f} | {row['best_iter']} -> {row['last_iter']} | "
        f"{row['peak_to_last_score_drop']:.2f} | {row['proxy_relative_return']:.2f}pp / {row['proxy_max_dd']:.2f}% | "
        f"{row['test_2022_max_dd']:.2f}% / {row['test_2022_avg_pos']:.3f} | {str(bool(row['walk_forward_eligible'])).lower()} |"
    )

overfit_lines = []
for _, row in overfit_examples.iterrows():
    overfit_lines.append(f"- `{row['profile']}`: best iter `{row['best_iter']}`, last iter `{row['last_iter']}`, score drop `{row['peak_to_last_score_drop']:.2f}`")
if not overfit_lines:
    overfit_lines = ['- no selection-history peak-to-last score drops above threshold']

strict_pass_count = int(strict_daily['target_pass'].sum())
strict_pass_rows = strict_daily[strict_daily['target_pass']].sort_values(['max_dd', 'relative_return'], ascending=[True, False])
strict_pass_lines = []
for _, row in strict_pass_rows.iterrows():
    strict_pass_lines.append(
        f"- `{row['candidate']}`: return `{row['return']:.2f}%`, relative `{row['relative_return']:.2f}pp`, "
        f"MaxDD `{row['max_dd']:.2f}%`, avg_pos `{row['avg_pos']:.3f}`, path `{row['path']}`"
    )
if not strict_pass_lines:
    strict_pass_lines = ['- no strict daily artifact reaches both relative >= +30pp and MaxDD <= 15%']

wf_gate_lines = []
residual_wf_selection_path = ROOT / 'runtime' / 'execution' / 'card' / 'risk_budget_residual_drawdown_guard_walkforward_v1' / 'selection.json'
if residual_wf_selection_path.exists():
    residual_wf = load_json(residual_wf_selection_path)
    wf_gate_lines.append(
        f"- `risk_budget_residual_drawdown_guard_v1` strict pass is linked to `{residual_wf.get('experiment')}`; decision `{residual_wf.get('decision')}`."
    )
    comparison_rel = residual_wf.get('artifacts', {}).get('comparison')
    if comparison_rel:
        comparison_path = ROOT / comparison_rel
        if comparison_path.exists():
            comparison = pd.read_csv(comparison_path)
            residual_rows = comparison.sort_values('window')
            bits = []
            for _, row in residual_rows.iterrows():
                bits.append(
                    f"{row['window']} MaxDD `{row['residual_max_dd']:.2f}%` / relative `{row['residual_relative_return']:.2f}pp` / pass `{str(bool(row['residual_target_pass'])).lower()}`"
                )
            wf_gate_lines.append('- Saved walk-forward residual guard: ' + '; '.join(bits) + '.')
if not wf_gate_lines:
    wf_gate_lines = ['- no linked walk-forward gate artifact found for strict-pass candidates; treat strict-only pass as not promotable']

conditional_rows = strict_daily[strict_daily['candidate'] == 'recovery_reentry_conditional_vol_target_v1']
mechanism_lines = []
if not conditional_rows.empty:
    row = conditional_rows.iloc[0]
    mechanism_lines.append(
        f"- `recovery_reentry_conditional_vol_target_v1` refreshed in this audit: relative `{row['relative_return']:.2f}pp`, MaxDD `{row['max_dd']:.2f}%`, target_pass `{str(bool(row['target_pass'])).lower()}`."
    )
if not mechanism_lines:
    mechanism_lines = ['- no newer cash-only overlay changed the selection frontier materially']

if strict_pass_count:
    decision_lines = [
        '- Diagnostic keep / no runtime promotion: the refreshed strict screen has at least one fixed-path target pass, but the known pass is blocked by saved walk-forward validation.',
        '- Treat full-window strict passes as candidate discovery only. Runtime promotion still requires the linked walk-forward notebook to pass all required windows.',
        '- The remaining high-return cash-only overlays still sit near `30%` strict MaxDD. The next useful mechanism should change the risk-budget objective/architecture or explicitly test a low-drawdown capital-allocation layer, rather than adding another threshold cap.',
    ]
else:
    decision_lines = [
        '- Diagnostic keep / no runtime promotion: the current repo has no existing strict daily artifact and no simple conservative risk-budget rescaling that reaches both `+30pp` excess and `<=15%` MaxDD.',
        '- The closest path is not continued PPO. It is the low-drawdown `risk_budget_usage_v1_actor_parity` family, but even conservative scaling/drawdown control cannot cross 15% while preserving excess.',
        '- This supports a tighter next experiment: change the risk-budget objective/architecture or add a true low-drawdown capital-allocation layer, rather than retraining v22, repeating SOXX features, or tuning broad execution caps.',
    ]

readout = [
    '## Final Readout',
    '',
    'This notebook is the primary single-run conclusion artifact.',
    '',
    f"- Summary JSONs scanned: `{len(summary_table)}`; proxy target pass (`relative >= 30pp` and `MaxDD <= 15%`): `{int(summary_table['proxy_target_pass'].sum())}`.",
    f"- Strict daily artifacts scanned: `{len(strict_daily)}`; strict target pass: `{int(strict_daily['target_pass'].sum())}`.",
    f"- Risk-budget conservative frontier rows: `{len(frontier)}`; frontier target pass: `{int(frontier['target_pass'].sum())}`.",
    f"- Closest strict daily artifact by target gap: `{strict_best['candidate']}` at return `{strict_best['return']:.2f}%`, relative `{strict_best['relative_return']:.2f}pp`, MaxDD `{strict_best['max_dd']:.2f}%`, path `{strict_best['path']}`.",
    f"- Lowest-MaxDD strict artifact with relative >= +30pp: `{strict_lowest_dd_rel_pass['candidate']}` at relative `{strict_lowest_dd_rel_pass['relative_return']:.2f}pp`, MaxDD `{strict_lowest_dd_rel_pass['max_dd']:.2f}%`.",
    f"- Closest conservative risk-budget point: method `{frontier_best['method']}`, relative `{frontier_best['relative_return']:.2f}pp`, MaxDD `{frontier_best['max_dd']:.2f}%`, target pass `{str(bool(frontier_best['target_pass'])).lower()}`.",
    f"- Lowest-MaxDD conservative risk-budget point with relative >= +30pp: method `{frontier_rel_pass['method']}`, relative `{frontier_rel_pass['relative_return']:.2f}pp`, MaxDD `{frontier_rel_pass['max_dd']:.2f}%`.",
    '',
    '### Strict-Pass Candidates',
    '',
    *strict_pass_lines,
    '',
    '### Walk-Forward Gate Notes',
    '',
    *wf_gate_lines,
    '',
    '### Recent Mechanism Notes',
    '',
    *mechanism_lines,
    '',
    '### Current Model-Selection Rows',
    '',
    '| profile | best_score | best -> last iter | score drop | proxy relative / MaxDD | 2022 MaxDD / avg_pos | WF eligible |',
    '| --- | ---: | ---: | ---: | ---: | ---: | ---: |',
    *summary_lines,
    '',
    '### Selection-Overfit Examples',
    '',
    *overfit_lines,
    '',
    '### Decision',
    '',
    *decision_lines,
]
display(Markdown('\n'.join(readout)))


## Final Readout

This notebook is the primary single-run conclusion artifact.

- Summary JSONs scanned: `94`; proxy target pass (`relative >= 30pp` and `MaxDD <= 15%`): `0`.
- Strict daily artifacts scanned: `35`; strict target pass: `1`.
- Risk-budget conservative frontier rows: `594`; frontier target pass: `0`.
- Closest strict daily artifact by target gap: `risk_budget_residual_drawdown_guard_v1` at return `176.60%`, relative `30.46pp`, MaxDD `14.64%`, path `runtime/execution/card/risk_budget_residual_drawdown_guard_v1/risk_budget_residual_drawdown_guard_v1_selected_soxl_daily.csv`.
- Lowest-MaxDD strict artifact with relative >= +30pp: `risk_budget_residual_drawdown_guard_v1` at relative `30.46pp`, MaxDD `14.64%`.
- Closest conservative risk-budget point: method `scaled_floor`, relative `37.37pp`, MaxDD `18.38%`, target pass `false`.
- Lowest-MaxDD conservative risk-budget point with relative >= +30pp: method `scaled_floor`, relative `37.37pp`, MaxDD `18.38%`.

### Strict-Pass Candidates

- `risk_budget_residual_drawdown_guard_v1`: return `176.60%`, relative `30.46pp`, MaxDD `14.64%`, avg_pos `0.161`, path `runtime/execution/card/risk_budget_residual_drawdown_guard_v1/risk_budget_residual_drawdown_guard_v1_selected_soxl_daily.csv`

### Walk-Forward Gate Notes

- `risk_budget_residual_drawdown_guard_v1` strict pass is linked to `risk_budget_residual_drawdown_guard_walkforward_v1`; decision `candidate_blocked_by_walkforward`.
- Saved walk-forward residual guard: wf_2020 MaxDD `18.10%` / relative `-54.19pp` / pass `false`; wf_2022 MaxDD `10.97%` / relative `76.66pp` / pass `true`; wf_2023_2024 MaxDD `17.23%` / relative `38.31pp` / pass `false`.

### Recent Mechanism Notes

- `recovery_reentry_conditional_vol_target_v1` refreshed in this audit: relative `2171.72pp`, MaxDD `30.92%`, target_pass `false`.

### Current Model-Selection Rows

| profile | best_score | best -> last iter | score drop | proxy relative / MaxDD | 2022 MaxDD / avg_pos | WF eligible |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| `balanced_rule_residual_v19_reg` | 93.50 | 40 -> 80 | 0.90 | nanpp / 47.67% | 30.23% / 0.250 | false |
| `balanced_rule_residual_v19_reg` | 87.08 | 40 -> 80 | 0.77 | nanpp / 20.78% | nan% / nan | false |
| `balanced_rule_residual_v19_reg` | 79.74 | 20 -> 80 | 12.38 | nanpp / 31.04% | 32.50% / 0.278 | false |
| `balanced_rule_residual_v19_reg` | 65.99 | 40 -> 80 | 3.61 | nanpp / 37.23% | nan% / nan | false |
| `balanced_rule_residual_v19_reg_light` | 90.22 | 60 -> 80 | 20.87 | nanpp / 53.36% | 31.41% / 0.260 | false |
| `balanced_rule_residual_v19_reg_light` | 86.05 | 20 -> 80 | 5.61 | nanpp / 20.09% | nan% / nan | false |
| `balanced_rule_residual_v19_reg_light` | 70.47 | 60 -> 80 | 15.38 | nanpp / 30.23% | 30.23% / 0.250 | false |
| `balanced_rule_residual_v19_reg_light` | 71.74 | 20 -> 80 | 9.36 | nanpp / 41.89% | nan% / nan | false |
| `balanced_rule_residual_v22` | 91.94 | 80 -> 80 | 0.00 | nanpp / 54.52% | 32.71% / 0.269 | true |
| `balanced_rule_residual_v22` | 89.96 | 60 -> 80 | 14.22 | nanpp / 26.62% | nan% / nan | false |
| `balanced_rule_residual_v22` | 75.43 | 80 -> 80 | 0.00 | nanpp / 31.69% | 31.77% / 0.262 | false |
| `balanced_rule_residual_v22` | 73.39 | 80 -> 80 | 0.00 | nanpp / 42.67% | nan% / nan | false |
| `balanced_rule_residual_v22_continued_retention_v1` | 86.24 | 0 -> 20 | 16.98 | -1084.29pp / 59.13% | 55.26% / 0.321 | false |

### Selection-Overfit Examples

- `balanced_floor40_sym_expectancy`: best iter `0`, last iter `80`, score drop `41.63`
- `balanced_floor40_sym_expectancy`: best iter `0`, last iter `40`, score drop `33.08`
- `balanced_rule_residual_minimal_v1`: best iter `0`, last iter `80`, score drop `31.73`
- `balanced_rule_residual_refined_v2`: best iter `60`, last iter `80`, score drop `30.32`
- `balanced_rule_residual_v19_small`: best iter `0`, last iter `80`, score drop `29.85`
- `balanced_rule_residual_imitation_v4`: best iter `0`, last iter `20`, score drop `28.56`
- `balanced_rule_residual_v20`: best iter `60`, last iter `80`, score drop `26.82`
- `balanced_rule_residual_refined_v4`: best iter `20`, last iter `80`, score drop `23.88`

### Decision

- Diagnostic keep / no runtime promotion: the refreshed strict screen has at least one fixed-path target pass, but the known pass is blocked by saved walk-forward validation.
- Treat full-window strict passes as candidate discovery only. Runtime promotion still requires the linked walk-forward notebook to pass all required windows.
- The remaining high-return cash-only overlays still sit near `30%` strict MaxDD. The next useful mechanism should change the risk-budget objective/architecture or explicitly test a low-drawdown capital-allocation layer, rather than adding another threshold cap.